In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
"""
IMPROVED SYSTEMATIC REVIEW ANALYSIS - AI for Communication in Dementia & Aphasia
=================================================================================

Key improvements:
1. Data cleaning to fix bracketed values and case sensitivity
2. Better visualization layouts with readable labels
3. Consistent handling of categorical variables
4. Improved color schemes and figure sizes
5. FIXED: Correct column names (singular not plural)
"""

import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# For clustering
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

print("✓ Packages loaded\n")

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Analysis configuration"""
    # File paths
    INPUT_CSV = "/kaggle/input/385reviewswithabstractsannotated301025/annotated_results.csv"
    INPUT_JSON = "/kaggle/input/385reviewswithabstractsannotated301025/annotations.json"
    
    # Output files
    OUTPUT_DIR = "analysis_outputs/"
    
    # Visualization settings
    FIGURE_SIZE = (12, 6)
    COLOR_PALETTE = "Set2"
    
    # Filtering thresholds
    MIN_PARTICIPANTS = 10
    FOCUS_YEARS = [2020, 2025]

# ============================================================================
# DATA LOADING AND CLEANING
# ============================================================================

def clean_bracketed_strings(value):
    """Remove brackets and quotes from string values"""
    if pd.isna(value):
        return value
    
    value = str(value)
    
    # Remove square brackets and quotes
    value = value.strip("[]'\"")
    
    # Handle parentheses (but keep them if they're part of the content)
    # Remove only leading/trailing parentheses
    if value.startswith('(') and value.endswith(')'):
        value = value[1:-1]
    
    return value.strip()

def clean_dataframe(df):
    """Clean all categorical columns in the dataframe"""
    print("Cleaning dataframe...")
    
    # List of columns that might have bracketed values
    categorical_cols = [
        'dementia_relevance', 'aphasia_relevance', 'ai_communication_relevance',
        'ai_involvement', 'study_design', 'primary_technology',
        'intervention_setting', 'dementia_type', 'aphasia_type',
        'communication_modality', 'communication_domain'  # FIXED: singular forms
    ]
    
    # Clean bracketed strings
    for col in categorical_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_bracketed_strings)
    
    # Standardize relevance values (ensure consistency)
    relevance_cols = ['dementia_relevance', 'aphasia_relevance', 'ai_communication_relevance']
    for col in relevance_cols:
        if col in df.columns:
            # Convert to lowercase and strip whitespace
            df[col] = df[col].str.lower().str.strip()
    
    # Clean list-based columns (split properly)
    list_cols = ['ai_techniques', 'study_focus']  # Removed non-existent columns
    for col in list_cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: clean_bracketed_strings(x) if pd.notna(x) else x)
    
    print(f"✓ Cleaned {len(categorical_cols)} categorical columns")
    return df

def load_data():
    """Load and clean annotated results"""
    print("Loading annotated data...")
    
    # Load CSV
    df = pd.read_csv(Config.INPUT_CSV)
    print(f"✓ Loaded {len(df)} annotated records")
    
    # Clean the dataframe
    df = clean_dataframe(df)
    
    # Load JSON for richer annotation data if needed
    try:
        with open(Config.INPUT_JSON, 'r') as f:
            annotations_json = json.load(f)
    except FileNotFoundError:
        print("⚠️  JSON file not found, continuing with CSV only")
        annotations_json = None
    
    # Basic data quality check
    if 'annotation_success' in df.columns:
        success_rate = df['annotation_success'].mean()
        print(f"✓ Annotation success rate: {success_rate:.1%}")
        
        if success_rate < 0.95:
            print(f"⚠️  Warning: {(1-success_rate)*100:.1f}% of annotations failed")
    
    return df, annotations_json

# ============================================================================
# PART 1: RELEVANCE ANALYSIS
# ============================================================================

def analyze_relevance(df):
    """Analyze relevance scores to understand dataset composition"""
    
    print("\n" + "="*80)
    print("PART 1: RELEVANCE ANALYSIS")
    print("="*80 + "\n")
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Relevance Distribution Across Review Topics', fontsize=16, fontweight='bold')
    
    # Define a consistent order for relevance categories
    relevance_order = ['highly_relevant', 'moderately_relevant', 'marginally_relevant', 'not_relevant', 'not_assessed']
    colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#95a5a6']
    
    # 1. Dementia Relevance
    dementia_counts = df['dementia_relevance'].value_counts()
    # Reindex to ensure consistent order
    dementia_counts = dementia_counts.reindex(relevance_order, fill_value=0)
    axes[0, 0].bar(range(len(dementia_counts)), dementia_counts.values, 
                   color=colors[:len(dementia_counts)], edgecolor='black', linewidth=1.5)
    axes[0, 0].set_title('Dementia Relevance', fontweight='bold', fontsize=12)
    axes[0, 0].set_ylabel('Number of Papers', fontsize=11)
    axes[0, 0].set_xticks(range(len(dementia_counts)))
    axes[0, 0].set_xticklabels([x.replace('_', ' ').title() for x in dementia_counts.index], 
                                rotation=45, ha='right', fontsize=9)
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # 2. Aphasia Relevance
    aphasia_counts = df['aphasia_relevance'].value_counts()
    aphasia_counts = aphasia_counts.reindex(relevance_order, fill_value=0)
    axes[0, 1].bar(range(len(aphasia_counts)), aphasia_counts.values, 
                   color=colors[:len(aphasia_counts)], edgecolor='black', linewidth=1.5)
    axes[0, 1].set_title('Aphasia Relevance', fontweight='bold', fontsize=12)
    axes[0, 1].set_ylabel('Number of Papers', fontsize=11)
    axes[0, 1].set_xticks(range(len(aphasia_counts)))
    axes[0, 1].set_xticklabels([x.replace('_', ' ').title() for x in aphasia_counts.index], 
                                rotation=45, ha='right', fontsize=9)
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    # 3. AI-Communication Relevance
    ai_counts = df['ai_communication_relevance'].value_counts()
    ai_counts = ai_counts.reindex(relevance_order, fill_value=0)
    axes[1, 0].bar(range(len(ai_counts)), ai_counts.values, 
                   color=colors[:len(ai_counts)], edgecolor='black', linewidth=1.5)
    axes[1, 0].set_title('AI-Communication Relevance', fontweight='bold', fontsize=12)
    axes[1, 0].set_ylabel('Number of Papers', fontsize=11)
    axes[1, 0].set_xticks(range(len(ai_counts)))
    axes[1, 0].set_xticklabels([x.replace('_', ' ').title() for x in ai_counts.index], 
                                rotation=45, ha='right', fontsize=9)
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # 4. Combined High Relevance
    high_relevance = df[
        (df['dementia_relevance'] == 'highly_relevant') |
        (df['aphasia_relevance'] == 'highly_relevant')
    ]
    
    high_ai = high_relevance['ai_communication_relevance'].value_counts()
    high_ai = high_ai.reindex(relevance_order, fill_value=0)
    axes[1, 1].bar(range(len(high_ai)), high_ai.values, 
                   color=colors[:len(high_ai)], edgecolor='black', linewidth=1.5)
    axes[1, 1].set_title('AI Relevance Among High Dementia/Aphasia Papers', fontweight='bold', fontsize=12)
    axes[1, 1].set_ylabel('Number of Papers', fontsize=11)
    axes[1, 1].set_xticks(range(len(high_ai)))
    axes[1, 1].set_xticklabels([x.replace('_', ' ').title() for x in high_ai.index], 
                                rotation=45, ha='right', fontsize=9)
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('01_relevance_overview.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print summary statistics
    print("\nRELEVANCE SUMMARY:")
    print("-" * 60)
    print(f"Highly relevant to dementia: {(df['dementia_relevance'] == 'highly_relevant').sum()}")
    print(f"Highly relevant to aphasia: {(df['aphasia_relevance'] == 'highly_relevant').sum()}")
    print(f"Highly relevant to AI-communication: {(df['ai_communication_relevance'] == 'highly_relevant').sum()}")
    print(f"\nPapers highly relevant to ALL THREE: {len(df[(df['dementia_relevance'] == 'highly_relevant') & (df['aphasia_relevance'] == 'highly_relevant') & (df['ai_communication_relevance'] == 'highly_relevant')])}") 
    print(f"Papers highly relevant to dementia + AI: {len(df[(df['dementia_relevance'] == 'highly_relevant') & (df['ai_communication_relevance'] == 'highly_relevant')])}") 
    print(f"Papers highly relevant to aphasia + AI: {len(df[(df['aphasia_relevance'] == 'highly_relevant') & (df['ai_communication_relevance'] == 'highly_relevant')])}")

# ============================================================================
# PART 2: AI INVOLVEMENT ANALYSIS
# ============================================================================

def analyze_ai_involvement(df):
    """Analyze levels and types of AI involvement"""
    
    print("\n" + "="*80)
    print("PART 2: AI INVOLVEMENT ANALYSIS")
    print("="*80 + "\n")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('AI Involvement Patterns', fontsize=16, fontweight='bold')
    
    # 1. AI Involvement Distribution (Pie Chart)
    ai_involvement = df['ai_involvement'].value_counts()
    
    # Define colors for consistency
    colors_pie = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#95a5a6', '#9b59b6']
    
    # Create pie chart with better labels
    labels = [x.replace('_', ' ').title() if pd.notna(x) else 'Unknown' for x in ai_involvement.index]
    wedges, texts, autotexts = axes[0].pie(ai_involvement.values, labels=labels, autopct='%1.1f%%',
                                             colors=colors_pie, startangle=90,
                                             textprops={'fontsize': 10})
    
    # Make percentage text bold
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(11)
    
    axes[0].set_title('AI Involvement Levels', fontweight='bold', fontsize=12)
    
    # 2. Top 10 AI Techniques
    # Parse AI techniques more carefully
    ai_tech_list = []
    for tech_str in df['ai_techniques'].dropna():
        if tech_str and str(tech_str).lower() not in ['not_specified', 'nan', 'none', '']:
            # Split by comma and clean
            techniques = [t.strip() for t in str(tech_str).split(',')]
            ai_tech_list.extend([t for t in techniques if len(t) > 2])
    
    if ai_tech_list:
        tech_counts = Counter(ai_tech_list).most_common(10)
        tech_names = [clean_bracketed_strings(t[0]) for t in tech_counts]
        tech_values = [t[1] for t in tech_counts]
        
        axes[1].barh(range(len(tech_names)), tech_values, color='#3498db', edgecolor='black', linewidth=1.5)
        axes[1].set_yticks(range(len(tech_names)))
        axes[1].set_yticklabels([t.replace('_', ' ').title() for t in tech_names], fontsize=10)
        axes[1].set_xlabel('Number of Papers', fontsize=11)
        axes[1].set_title('Top 10 AI Techniques Mentioned', fontweight='bold', fontsize=12)
        axes[1].grid(axis='x', alpha=0.3)
        axes[1].invert_yaxis()
    else:
        axes[1].text(0.5, 0.5, 'No AI techniques specified', 
                    ha='center', va='center', fontsize=12)
        axes[1].set_title('Top 10 AI Techniques Mentioned', fontweight='bold', fontsize=12)
    
    plt.tight_layout()
    plt.savefig('02_ai_involvement.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print statistics
    print("\nAI INVOLVEMENT BREAKDOWN:")
    print("-" * 60)
    for level, count in ai_involvement.items():
        print(f"{level}: {count} papers ({count/len(df)*100:.1f}%)")

# ============================================================================
# PART 3: TECHNOLOGY LANDSCAPE  
# ============================================================================

def analyze_technology_landscape(df):
    """Analyze technology types and communication aspects"""
    
    print("\n" + "="*80)
    print("PART 3: TECHNOLOGY & COMMUNICATION LANDSCAPE")
    print("="*80 + "\n")
    
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])
    
    fig.suptitle('Technology Types and Communication Modalities', fontsize=16, fontweight='bold')
    
    # 1. Technology Types (Horizontal bar chart, top 15)
    tech_list = []
    for tech_str in df['primary_technology'].dropna():
        if tech_str and str(tech_str).lower() not in ['not_specified', 'nan', 'none', '']:
            techs = [t.strip() for t in str(tech_str).split(',')]
            tech_list.extend([t for t in techs if len(t) > 2])
    
    if tech_list:
        tech_counts = Counter(tech_list).most_common(15)
        tech_names = [clean_bracketed_strings(t[0]) for t in tech_counts]
        tech_values = [t[1] for t in tech_counts]
        
        ax1.barh(range(len(tech_names)), tech_values, color='#1abc9c', edgecolor='black', linewidth=1.5)
        ax1.set_yticks(range(len(tech_names)))
        ax1.set_yticklabels([t.replace('_', ' ').title() for t in tech_names], fontsize=9)
        ax1.set_xlabel('Number of Papers', fontsize=10)
        ax1.set_title('Technology Types (Top 15)', fontweight='bold', fontsize=11)
        ax1.grid(axis='x', alpha=0.3)
        ax1.invert_yaxis()
    else:
        ax1.text(0.5, 0.5, 'No technology data available', 
                ha='center', va='center', fontsize=12, transform=ax1.transAxes)
        ax1.set_title('Technology Types (Top 15)', fontweight='bold', fontsize=11)
    
    # 2. Communication Modalities (FIXED: singular column name)
    comm_mod_list = []
    if 'communication_modality' in df.columns:
        for mod_str in df['communication_modality'].dropna():
            if mod_str and str(mod_str).lower() not in ['not_specified', 'nan', 'none', '']:
                mods = [m.strip() for m in str(mod_str).split(',') if len(m.strip()) > 2]
                comm_mod_list.extend(mods)
    
    if comm_mod_list:
        mod_counts = Counter(comm_mod_list).most_common(10)
        mod_names = [clean_bracketed_strings(m[0]) for m in mod_counts]
        mod_values = [m[1] for m in mod_counts]
        
        ax2.bar(range(len(mod_names)), mod_values, color='#e67e22', edgecolor='black', linewidth=1.5)
        ax2.set_xticks(range(len(mod_names)))
        ax2.set_xticklabels([m.replace('_', ' ').title() for m in mod_names], 
                            rotation=45, ha='right', fontsize=9)
        ax2.set_ylabel('Number of Papers', fontsize=10)
        ax2.set_title('Communication Modalities (Top 10)', fontweight='bold', fontsize=11)
        ax2.grid(axis='y', alpha=0.3)
    else:
        ax2.text(0.5, 0.5, 'No communication modality data', 
                ha='center', va='center', fontsize=12, transform=ax2.transAxes)
        ax2.set_title('Communication Modalities (Top 10)', fontweight='bold', fontsize=11)
    
    # 3. Technology by AI Involvement
    tech_by_ai = df.groupby('ai_involvement')['primary_technology'].count()
    ai_order = ['central_focus', 'supporting_feature', 'mentioned_only', 'none', 'unclear']
    tech_by_ai = tech_by_ai.reindex([x for x in ai_order if x in tech_by_ai.index], fill_value=0)
    
    if len(tech_by_ai) > 0:
        ax3.bar(range(len(tech_by_ai)), tech_by_ai.values, color='#9b59b6', edgecolor='black', linewidth=1.5)
        ax3.set_xticks(range(len(tech_by_ai)))
        ax3.set_xticklabels([x.replace('_', ' ').title() for x in tech_by_ai.index], 
                            rotation=45, ha='right', fontsize=9)
        ax3.set_ylabel('Number of Technology Mentions', fontsize=10)
        ax3.set_title('Technology Mentions by AI Involvement', fontweight='bold', fontsize=11)
        ax3.grid(axis='y', alpha=0.3)
    
    # 4. Communication Domains (FIXED: singular column name)
    comm_dom_list = []
    if 'communication_domain' in df.columns:
        for dom_str in df['communication_domain'].dropna():
            if dom_str and str(dom_str).lower() not in ['not_specified', 'nan', 'none', '']:
                doms = [d.strip() for d in str(dom_str).split(',') if len(d.strip()) > 2]
                comm_dom_list.extend(doms)
    
    if comm_dom_list:
        dom_counts = Counter(comm_dom_list).most_common(12)
        dom_names = [clean_bracketed_strings(d[0]) for d in dom_counts]
        dom_values = [d[1] for d in dom_counts]
        
        ax4.barh(range(len(dom_names)), dom_values, color='#f39c12', edgecolor='black', linewidth=1.5)
        ax4.set_yticks(range(len(dom_names)))
        ax4.set_yticklabels([d.replace('_', ' ').title() for d in dom_names], fontsize=9)
        ax4.set_xlabel('Number of Papers', fontsize=10)
        ax4.set_title('Communication Domains (Top 12)', fontweight='bold', fontsize=11)
        ax4.grid(axis='x', alpha=0.3)
        ax4.invert_yaxis()
    else:
        ax4.text(0.5, 0.5, 'No communication domain data', 
                ha='center', va='center', fontsize=12, transform=ax4.transAxes)
        ax4.set_title('Communication Domains (Top 12)', fontweight='bold', fontsize=11)
    
    plt.savefig('03_technology_landscape.png', dpi=300, bbox_inches='tight')
    plt.show()

# ============================================================================
# PART 4: STUDY CHARACTERISTICS
# ============================================================================

def analyze_study_characteristics(df):
    """Analyze study designs and settings"""
    
    print("\n" + "="*80)
    print("PART 4: STUDY CHARACTERISTICS")
    print("="*80 + "\n")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Study Design and Setting Analysis', fontsize=16, fontweight='bold')
    
    # 1. Study Design Distribution
    if 'study_design' in df.columns:
        design_counts = df['study_design'].value_counts().head(10)
        axes[0, 0].barh(range(len(design_counts)), design_counts.values, 
                       color='#3498db', edgecolor='black', linewidth=1.5)
        axes[0, 0].set_yticks(range(len(design_counts)))
        axes[0, 0].set_yticklabels([clean_bracketed_strings(d).replace('_', ' ').title() 
                                    for d in design_counts.index], fontsize=9)
        axes[0, 0].set_xlabel('Number of Papers', fontsize=10)
        axes[0, 0].set_title('Study Designs (Top 10)', fontweight='bold', fontsize=11)
        axes[0, 0].grid(axis='x', alpha=0.3)
        axes[0, 0].invert_yaxis()
    
    # 2. Intervention Setting
    if 'intervention_setting' in df.columns:
        setting_counts = df['intervention_setting'].value_counts().head(10)
        axes[0, 1].bar(range(len(setting_counts)), setting_counts.values, 
                      color='#2ecc71', edgecolor='black', linewidth=1.5)
        axes[0, 1].set_xticks(range(len(setting_counts)))
        axes[0, 1].set_xticklabels([clean_bracketed_strings(s).replace('_', ' ').title() 
                                    for s in setting_counts.index], 
                                   rotation=45, ha='right', fontsize=9)
        axes[0, 1].set_ylabel('Number of Papers', fontsize=10)
        axes[0, 1].set_title('Intervention Settings (Top 10)', fontweight='bold', fontsize=11)
        axes[0, 1].grid(axis='y', alpha=0.3)
    
    # 3. Publication Year Distribution
    if 'year' in df.columns:
        year_counts = df['year'].value_counts().sort_index()
        axes[1, 0].plot(year_counts.index, year_counts.values, 
                       marker='o', linewidth=2, markersize=8, color='#e74c3c')
        axes[1, 0].fill_between(year_counts.index, year_counts.values, alpha=0.3, color='#e74c3c')
        axes[1, 0].set_xlabel('Year', fontsize=10)
        axes[1, 0].set_ylabel('Number of Papers', fontsize=10)
        axes[1, 0].set_title('Publication Year Distribution', fontweight='bold', fontsize=11)
        axes[1, 0].grid(alpha=0.3)
    
    # 4. Study Focus Areas
    if 'study_focus' in df.columns:
        focus_list = []
        for focus_str in df['study_focus'].dropna():
            if focus_str and str(focus_str).lower() not in ['not_specified', 'nan', 'none', '']:
                focuses = [f.strip() for f in str(focus_str).split(',') if len(f.strip()) > 2]
                focus_list.extend(focuses)
        
        if focus_list:
            focus_counts = Counter(focus_list).most_common(10)
            focus_names = [clean_bracketed_strings(f[0]) for f in focus_counts]
            focus_values = [f[1] for f in focus_counts]
            
            axes[1, 1].barh(range(len(focus_names)), focus_values, 
                           color='#f39c12', edgecolor='black', linewidth=1.5)
            axes[1, 1].set_yticks(range(len(focus_names)))
            axes[1, 1].set_yticklabels([f.replace('_', ' ').title() for f in focus_names], fontsize=9)
            axes[1, 1].set_xlabel('Number of Papers', fontsize=10)
            axes[1, 1].set_title('Study Focus Areas (Top 10)', fontweight='bold', fontsize=11)
            axes[1, 1].grid(axis='x', alpha=0.3)
            axes[1, 1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('04_study_characteristics.png', dpi=300, bbox_inches='tight')
    plt.show()

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Load and clean data
    df, annotations = load_data()
    
    # Run analyses
    analyze_relevance(df)
    analyze_ai_involvement(df)
    analyze_technology_landscape(df)
    analyze_study_characteristics(df)
    
    print("\n" + "="*80)
    print("✓ ANALYSIS COMPLETE!")
    print("="*80)
    print("\nGenerated visualizations:")
    print("  - 01_relevance_overview.png")
    print("  - 02_ai_involvement.png")
    print("  - 03_technology_landscape.png")
    print("  - 04_study_characteristics.png")

In [1]:
"""
OPTION A FILTERING - KAGGLE VERSION
====================================
Optimized for Kaggle notebooks with proper paths
"""

# ============================================================================
# STEP 1: FIX PANDAS IMPORT (Kaggle environment issue)
# ============================================================================

# Restart kernel if needed, or just run this cell fresh
import pandas as pd
import os
from collections import Counter

print("="*80)
print("OPTION A: AI-FOCUSED FILTERING (Kaggle Edition)")
print("="*80)

# ============================================================================
# CONFIGURATION - KAGGLE PATHS
# ============================================================================

# Your file is in Kaggle input directory
INPUT_FILE = '/kaggle/input/385reviewswithabstractsannotated301025/annotated_results.csv'

print(f"\n✓ Loading from: {INPUT_FILE}")
print(f"  Current directory: {os.getcwd()}")

# ============================================================================
# LOAD DATA
# ============================================================================

print("\n[1/8] Loading data...")
df = pd.read_csv(INPUT_FILE)
print(f"  ✓ Loaded {len(df)} records")

# Verify columns
required_cols = ['record_id', 'title', 'ai_involvement', 
                 'dementia_relevance', 'aphasia_relevance', 
                 'ai_communication_relevance']

missing = [col for col in required_cols if col not in df.columns]
if missing:
    print(f"  ✗ Missing columns: {missing}")
    raise ValueError(f"Missing required columns: {missing}")

print(f"  ✓ All required columns present")

# ============================================================================
# TIER A: CONFIRMED AI (NO ai_communication filter)
# ============================================================================

print("\n[2/8] Filtering TIER A (Confirmed AI)...")
print("  Criteria:")
print("    • ai_involvement IN ('central_focus', 'supporting_feature')")
print("    • dementia/aphasia highly OR moderately relevant")
print("    • NO FILTER on ai_communication_relevance ← KEY DIFFERENCE")

tier_a = df[
    (df['ai_involvement'].isin(['central_focus', 'supporting_feature'])) &
    (
        (df['dementia_relevance'].isin(['highly_relevant', 'moderately_relevant'])) |
        (df['aphasia_relevance'].isin(['highly_relevant', 'moderately_relevant']))
    )
].copy()

print(f"\n  ✓ TIER A: {len(tier_a)} papers")
print(f"    - AI central: {(tier_a['ai_involvement'] == 'central_focus').sum()}")
print(f"    - AI supporting: {(tier_a['ai_involvement'] == 'supporting_feature').sum()}")

# ============================================================================
# TIER B: UNCLEAR AI - FOR MANUAL REVIEW
# ============================================================================

print("\n[3/8] Filtering TIER B (Unclear AI - needs manual review)...")

tier_b = df[
    (df['ai_involvement'] == 'unclear') &
    (
        (df['dementia_relevance'] == 'highly_relevant') |
        (df['aphasia_relevance'] == 'highly_relevant')
    )
].copy()

print(f"  ✓ TIER B: {len(tier_b)} papers")

# ============================================================================
# COMPARISON TO ORIGINAL APPROACH
# ============================================================================

print("\n[4/8] Comparing to original approach...")

original_tier2 = df[
    (df['ai_involvement'].isin(['central_focus', 'supporting_feature'])) &
    (
        (df['dementia_relevance'].isin(['highly_relevant', 'moderately_relevant'])) |
        (df['aphasia_relevance'].isin(['highly_relevant', 'moderately_relevant']))
    ) &
    # The filter we removed:
    (df['ai_communication_relevance'].isin(['highly_relevant', 'moderately_relevant']))
]

print(f"  Original TIER 2 (with ai_communication filter): {len(original_tier2)} papers")
print(f"  Option A TIER A (without filter): {len(tier_a)} papers")
print(f"  Difference: +{len(tier_a) - len(original_tier2)} papers")

# Show papers that were added
tier_a_ids = set(tier_a['record_id'])
original_ids = set(original_tier2['record_id'])
new_papers = tier_a[~tier_a['record_id'].isin(original_ids)]

if len(new_papers) > 0:
    print(f"\n  Papers ADDED by removing ai_communication filter:")
    for idx, row in new_papers.iterrows():
        print(f"    • {row['title'][:70]}...")
        print(f"      ai_communication: {row['ai_communication_relevance']}")

# ============================================================================
# PRIORITIZATION
# ============================================================================

print("\n[5/8] Creating prioritization subsets...")

priority_high = tier_a[tier_a['ai_communication_relevance'] == 'highly_relevant']
priority_medium = tier_a[tier_a['ai_communication_relevance'] == 'moderately_relevant']
priority_low = tier_a[tier_a['ai_communication_relevance'].isin(['marginally_relevant', 'not_relevant'])]

print(f"  HIGH priority: {len(priority_high)} papers")
print(f"  MEDIUM priority: {len(priority_medium)} papers")
print(f"  LOW priority: {len(priority_low)} papers")

# ============================================================================
# POPULATION BREAKDOWN
# ============================================================================

print("\n[6/8] Creating population-specific subsets...")

dementia_focused = tier_a[
    (tier_a['dementia_relevance'].isin(['highly_relevant', 'moderately_relevant'])) &
    (tier_a['aphasia_relevance'].isin(['not_relevant', 'marginally_relevant']))
]

aphasia_focused = tier_a[
    (tier_a['aphasia_relevance'].isin(['highly_relevant', 'moderately_relevant'])) &
    (tier_a['dementia_relevance'].isin(['not_relevant', 'marginally_relevant']))
]

both_populations = tier_a[
    (tier_a['dementia_relevance'].isin(['highly_relevant', 'moderately_relevant'])) &
    (tier_a['aphasia_relevance'].isin(['highly_relevant', 'moderately_relevant']))
]

print(f"  Dementia-focused: {len(dementia_focused)} papers")
print(f"  Aphasia-focused: {len(aphasia_focused)} papers")
print(f"  Both populations: {len(both_populations)} papers")

# ============================================================================
# CREATE MANUAL REVIEW CHECKLIST
# ============================================================================

print("\n[7/8] Creating manual review checklist for TIER B...")

checklist_cols = [
    'record_id', 'title', 'year', 'journal',
    'primary_technology', 'specific_device',
    'dementia_relevance', 'aphasia_relevance',
    'ai_communication_relevance', 'extraction_notes'
]

available_cols = [col for col in checklist_cols if col in tier_b.columns]
tier_b_checklist = tier_b[available_cols].copy()

# Add review columns
tier_b_checklist.insert(1, 'AI_CONFIRMED', '')
tier_b_checklist.insert(2, 'AI_EVIDENCE', '')
tier_b_checklist.insert(3, 'INCLUDE_IN_REVIEW', '')
tier_b_checklist.insert(4, 'REVIEWER_NOTES', '')

print(f"  ✓ Checklist created with {len(tier_b_checklist)} papers")

# ============================================================================
# SAVE FILES
# ============================================================================

print("\n[8/8] Saving output files...")

# In Kaggle, save to /kaggle/working (the output directory)
output_dir = '/kaggle/working'

try:
    # Main files
    tier_a.to_csv(f'{output_dir}/OPTION_A_TIER_A_confirmed_ai.csv', index=False)
    print(f"  ✓ OPTION_A_TIER_A_confirmed_ai.csv ({len(tier_a)} papers)")
    
    tier_b.to_csv(f'{output_dir}/OPTION_A_TIER_B_unclear_ai.csv', index=False)
    print(f"  ✓ OPTION_A_TIER_B_unclear_ai.csv ({len(tier_b)} papers)")
    
    tier_b_checklist.to_csv(f'{output_dir}/OPTION_A_TIER_B_CHECKLIST.csv', index=False)
    print(f"  ✓ OPTION_A_TIER_B_CHECKLIST.csv ({len(tier_b)} papers)")
    
    # Prioritization files
    if len(priority_high) > 0:
        priority_high.to_csv(f'{output_dir}/OPTION_A_HIGH_PRIORITY.csv', index=False)
        print(f"  ✓ OPTION_A_HIGH_PRIORITY.csv ({len(priority_high)} papers)")
    
    if len(priority_medium) > 0:
        priority_medium.to_csv(f'{output_dir}/OPTION_A_MEDIUM_PRIORITY.csv', index=False)
        print(f"  ✓ OPTION_A_MEDIUM_PRIORITY.csv ({len(priority_medium)} papers)")
    
    if len(priority_low) > 0:
        priority_low.to_csv(f'{output_dir}/OPTION_A_LOW_PRIORITY.csv', index=False)
        print(f"  ✓ OPTION_A_LOW_PRIORITY.csv ({len(priority_low)} papers)")
    
    # Population files
    if len(dementia_focused) > 0:
        dementia_focused.to_csv(f'{output_dir}/OPTION_A_dementia_focused.csv', index=False)
        print(f"  ✓ OPTION_A_dementia_focused.csv ({len(dementia_focused)} papers)")
    
    if len(aphasia_focused) > 0:
        aphasia_focused.to_csv(f'{output_dir}/OPTION_A_aphasia_focused.csv', index=False)
        print(f"  ✓ OPTION_A_aphasia_focused.csv ({len(aphasia_focused)} papers)")
    
    if len(both_populations) > 0:
        both_populations.to_csv(f'{output_dir}/OPTION_A_both_populations.csv', index=False)
        print(f"  ✓ OPTION_A_both_populations.csv ({len(both_populations)} papers)")
    
    print(f"\n  All files saved to: {output_dir}")

except Exception as e:
    print(f"  ✗ Error saving files: {str(e)}")

# ============================================================================
# SUMMARY & NEXT STEPS
# ============================================================================

print("\n" + "="*80)
print("✅ SUCCESS!")
print("="*80)

expected_tier_b = int(len(tier_b) * 0.3)
total_expected = len(tier_a) + expected_tier_b

print(f"""
📊 RESULTS:
   • TIER A (confirmed AI): {len(tier_a)} papers
   • TIER B (unclear AI): {len(tier_b)} papers
   • Expected after manual review: ~{expected_tier_b} papers
   • TOTAL for full-text screening: ~{total_expected} papers

📁 FILES CREATED:
   • OPTION_A_TIER_A_confirmed_ai.csv - Include ALL in screening
   • OPTION_A_TIER_B_CHECKLIST.csv - Manual review needed
   • Priority and population subsets

🔄 NEXT STEPS:

1. DOWNLOAD ALL FILES
   Click the folder icon (📁) in the left sidebar
   Download all OPTION_A_* files

2. FULL-TEXT SCREENING - START WITH TIER A ({len(tier_a)} papers)
   Include ALL papers in full-text screening
   Start with HIGH priority if you want

3. MANUAL REVIEW - TIER B CHECKLIST ({len(tier_b)} papers)
   Open OPTION_A_TIER_B_CHECKLIST.csv in Excel/Sheets
   Budget: 3-4 hours
   For each paper:
      • Look for AI signals (ML, NLP, recognition, adaptive)
      • Mark AI_CONFIRMED: YES/NO/MAYBE
      • Mark INCLUDE_IN_REVIEW: YES/NO
   Expected: ~{expected_tier_b} papers with confirmed AI

4. COMBINE FOR FULL-TEXT SCREENING
   TIER A ({len(tier_a)}) + TIER B confirmed (~{expected_tier_b}) = ~{total_expected} papers

5. FULL-TEXT SCREENING CRITERIA
   INCLUDE if:
   ✓ Uses AI/ML techniques
   ✓ Targets dementia OR aphasia
   ✓ Has communication relevance (direct or indirect)
   ✓ Peer-reviewed, full text available
   
   EXCLUDE if:
   ✗ No AI (just digital technology)
   ✗ Wrong population
   ✗ Protocol only
   ✗ Zero communication relevance

📈 EXPECTED FINAL YIELD:
   After full-text exclusions: ~{int(total_expected * 0.6)}-{int(total_expected * 0.8)} papers
   Ready for synthesis and analysis
""")

print("="*80)
print("🎉 FILTERING COMPLETE!")
print("="*80)

# Display first few papers from TIER A
print(f"\n📋 Preview of TIER A papers:")
print("-" * 80)
for idx, row in tier_a.head(5).iterrows():
    print(f"\n{idx+1}. {row['title'][:70]}...")
    print(f"   ai_involvement: {row['ai_involvement']}")
    print(f"   ai_communication: {row['ai_communication_relevance']}")
    print(f"   population: dementia={row['dementia_relevance']}, aphasia={row['aphasia_relevance']}")

if len(tier_a) > 5:
    print(f"\n... and {len(tier_a) - 5} more papers")

print("\n" + "="*80)


OPTION A: AI-FOCUSED FILTERING (Kaggle Edition)

✓ Loading from: /kaggle/input/385reviewswithabstractsannotated301025/annotated_results.csv
  Current directory: /kaggle/working

[1/8] Loading data...
  ✓ Loaded 385 records
  ✓ All required columns present

[2/8] Filtering TIER A (Confirmed AI)...
  Criteria:
    • ai_involvement IN ('central_focus', 'supporting_feature')
    • dementia/aphasia highly OR moderately relevant
    • NO FILTER on ai_communication_relevance ← KEY DIFFERENCE

  ✓ TIER A: 13 papers
    - AI central: 2
    - AI supporting: 11

[3/8] Filtering TIER B (Unclear AI - needs manual review)...
  ✓ TIER B: 39 papers

[4/8] Comparing to original approach...
  Original TIER 2 (with ai_communication filter): 12 papers
  Option A TIER A (without filter): 13 papers
  Difference: +1 papers

  Papers ADDED by removing ai_communication filter:
    • Effects of non-facilitated meaningful activities for people with demen...
      ai_communication: marginally_relevant

[5/8] Crea

In [ ]:
"""
OPTION A FILTERING - KAGGLE VERSION WITH HUMAN-READABLE OUTPUTS
================================================================
Creates CSV files PLUS easy-to-read HTML and TXT formats
"""

import pandas as pd
import os
from collections import Counter
from datetime import datetime

print("="*80)
print("OPTION A: AI-FOCUSED FILTERING (with readable outputs)")
print("="*80)

# ============================================================================
# LOAD DATA
# ============================================================================

print("\n[1/9] Loading data...")
INPUT_FILE = '/kaggle/input/385reviewswithabstractsannotated301025/annotated_results.csv'
df = pd.read_csv(INPUT_FILE)
print(f"  ✓ Loaded {len(df)} records")

# ============================================================================
# TIER A: CONFIRMED AI (NO ai_communication filter)
# ============================================================================

print("\n[2/9] Filtering TIER A (Confirmed AI)...")
print("  KEY: NO FILTER on ai_communication_relevance")

tier_a = df[
    (df['ai_involvement'].isin(['central_focus', 'supporting_feature'])) &
    (
        (df['dementia_relevance'].isin(['highly_relevant', 'moderately_relevant'])) |
        (df['aphasia_relevance'].isin(['highly_relevant', 'moderately_relevant']))
    )
].copy()

print(f"  ✓ TIER A: {len(tier_a)} papers")

# ============================================================================
# TIER B: UNCLEAR AI
# ============================================================================

print("\n[3/9] Filtering TIER B (Unclear AI)...")

tier_b = df[
    (df['ai_involvement'] == 'unclear') &
    (
        (df['dementia_relevance'] == 'highly_relevant') |
        (df['aphasia_relevance'] == 'highly_relevant')
    )
].copy()

print(f"  ✓ TIER B: {len(tier_b)} papers")

# ============================================================================
# COMPARISON
# ============================================================================

print("\n[4/9] Comparing to original approach...")

original = df[
    (df['ai_involvement'].isin(['central_focus', 'supporting_feature'])) &
    (
        (df['dementia_relevance'].isin(['highly_relevant', 'moderately_relevant'])) |
        (df['aphasia_relevance'].isin(['highly_relevant', 'moderately_relevant']))
    ) &
    (df['ai_communication_relevance'].isin(['highly_relevant', 'moderately_relevant']))
]

print(f"  Original: {len(original)} | Option A: {len(tier_a)} | Diff: +{len(tier_a)-len(original)}")

# ============================================================================
# PRIORITIZATION & SUBSETS
# ============================================================================

print("\n[5/9] Creating subsets...")

# Prioritization
priority_high = tier_a[tier_a['ai_communication_relevance'] == 'highly_relevant']
priority_medium = tier_a[tier_a['ai_communication_relevance'] == 'moderately_relevant']
priority_low = tier_a[tier_a['ai_communication_relevance'].isin(['marginally_relevant', 'not_relevant'])]

# Population
dementia = tier_a[
    (tier_a['dementia_relevance'].isin(['highly_relevant', 'moderately_relevant'])) &
    (tier_a['aphasia_relevance'].isin(['not_relevant', 'marginally_relevant']))
]
aphasia = tier_a[
    (tier_a['aphasia_relevance'].isin(['highly_relevant', 'moderately_relevant'])) &
    (tier_a['dementia_relevance'].isin(['not_relevant', 'marginally_relevant']))
]
both = tier_a[
    (tier_a['dementia_relevance'].isin(['highly_relevant', 'moderately_relevant'])) &
    (tier_a['aphasia_relevance'].isin(['highly_relevant', 'moderately_relevant']))
]

print(f"  Priority: HIGH={len(priority_high)}, MED={len(priority_medium)}, LOW={len(priority_low)}")
print(f"  Population: Dementia={len(dementia)}, Aphasia={len(aphasia)}, Both={len(both)}")

# ============================================================================
# CREATE CHECKLIST
# ============================================================================

print("\n[6/9] Creating manual review checklist...")

cols = ['record_id', 'title', 'year', 'primary_technology', 
        'dementia_relevance', 'aphasia_relevance', 
        'ai_communication_relevance', 'extraction_notes']
available = [c for c in cols if c in tier_b.columns]

checklist = tier_b[available].copy()
checklist.insert(1, 'AI_CONFIRMED', '')
checklist.insert(2, 'AI_EVIDENCE', '')
checklist.insert(3, 'INCLUDE', '')

# ============================================================================
# SAVE CSV FILES
# ============================================================================

print("\n[7/9] Saving CSV files...")

tier_a.to_csv('OPTION_A_TIER_A_confirmed_ai.csv', index=False)
tier_b.to_csv('OPTION_A_TIER_B_unclear_ai.csv', index=False)
checklist.to_csv('OPTION_A_TIER_B_CHECKLIST.csv', index=False)

if len(priority_high) > 0:
    priority_high.to_csv('OPTION_A_HIGH_PRIORITY.csv', index=False)
if len(priority_medium) > 0:
    priority_medium.to_csv('OPTION_A_MEDIUM_PRIORITY.csv', index=False)
if len(dementia) > 0:
    dementia.to_csv('OPTION_A_dementia_focused.csv', index=False)
if len(aphasia) > 0:
    aphasia.to_csv('OPTION_A_aphasia_focused.csv', index=False)

print("  ✓ CSV files saved")

# ============================================================================
# CREATE HUMAN-READABLE OUTPUTS
# ============================================================================

print("\n[8/9] Creating human-readable outputs...")

# ----------------------------------------------------------------------------
# 1. HTML REPORT (Can be opened in any browser)
# ----------------------------------------------------------------------------

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Option A Filtering Results</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            max-width: 1200px;
            margin: 20px auto;
            padding: 20px;
            background-color: #f5f5f5;
        }}
        h1 {{
            color: #2c3e50;
            border-bottom: 3px solid #3498db;
            padding-bottom: 10px;
        }}
        h2 {{
            color: #34495e;
            margin-top: 30px;
            border-left: 4px solid #3498db;
            padding-left: 10px;
        }}
        .summary-box {{
            background-color: white;
            border: 1px solid #ddd;
            border-radius: 5px;
            padding: 20px;
            margin: 20px 0;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        .stat {{
            display: inline-block;
            margin: 10px 20px 10px 0;
            padding: 10px 15px;
            background-color: #3498db;
            color: white;
            border-radius: 4px;
            font-weight: bold;
        }}
        .paper {{
            background-color: white;
            border-left: 4px solid #3498db;
            padding: 15px;
            margin: 15px 0;
            border-radius: 4px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        }}
        .paper-title {{
            font-size: 16px;
            font-weight: bold;
            color: #2c3e50;
            margin-bottom: 8px;
        }}
        .paper-details {{
            font-size: 14px;
            color: #666;
            margin: 5px 0;
        }}
        .badge {{
            display: inline-block;
            padding: 3px 8px;
            border-radius: 3px;
            font-size: 12px;
            font-weight: bold;
            margin-right: 5px;
        }}
        .badge-high {{ background-color: #e74c3c; color: white; }}
        .badge-medium {{ background-color: #f39c12; color: white; }}
        .badge-low {{ background-color: #95a5a6; color: white; }}
        .badge-dementia {{ background-color: #9b59b6; color: white; }}
        .badge-aphasia {{ background-color: #1abc9c; color: white; }}
        .badge-both {{ background-color: #e67e22; color: white; }}
        .next-steps {{
            background-color: #fff3cd;
            border: 1px solid #ffc107;
            border-radius: 5px;
            padding: 20px;
            margin: 20px 0;
        }}
        .checklist-paper {{
            background-color: #fff9e6;
            border-left: 4px solid #ffc107;
            padding: 15px;
            margin: 15px 0;
            border-radius: 4px;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            margin: 20px 0;
            background-color: white;
        }}
        th, td {{
            padding: 12px;
            text-align: left;
            border-bottom: 1px solid #ddd;
        }}
        th {{
            background-color: #3498db;
            color: white;
            font-weight: bold;
        }}
        tr:hover {{
            background-color: #f5f5f5;
        }}
    </style>
</head>
<body>
    <h1>🔬 Option A Filtering Results</h1>
    
    <div class="summary-box">
        <h2>📊 Summary Statistics</h2>
        <div class="stat">TIER A: {len(tier_a)} papers</div>
        <div class="stat">TIER B: {len(tier_b)} papers</div>
        <div class="stat">Original: {len(original)} papers</div>
        <div class="stat">Difference: +{len(tier_a) - len(original)}</div>
        
        <h3>By Priority (TIER A)</h3>
        <div class="stat" style="background-color: #e74c3c;">HIGH: {len(priority_high)}</div>
        <div class="stat" style="background-color: #f39c12;">MEDIUM: {len(priority_medium)}</div>
        <div class="stat" style="background-color: #95a5a6;">LOW: {len(priority_low)}</div>
        
        <h3>By Population (TIER A)</h3>
        <div class="stat" style="background-color: #9b59b6;">Dementia: {len(dementia)}</div>
        <div class="stat" style="background-color: #1abc9c;">Aphasia: {len(aphasia)}</div>
        <div class="stat" style="background-color: #e67e22;">Both: {len(both)}</div>
    </div>
    
    <div class="next-steps">
        <h2>🔄 Next Steps</h2>
        <ol>
            <li><strong>Full-text screening:</strong> Include ALL {len(tier_a)} TIER A papers</li>
            <li><strong>Manual review:</strong> Review {len(tier_b)} TIER B papers using checklist (budget 3-4 hours)</li>
            <li><strong>Expected outcome:</strong> ~{int(len(tier_b) * 0.3)} additional papers from TIER B</li>
            <li><strong>Total for screening:</strong> ~{len(tier_a) + int(len(tier_b) * 0.3)} papers</li>
        </ol>
    </div>
    
    <h2>📄 TIER A Papers (Confirmed AI - Include in Full-Text Screening)</h2>
    <p><strong>All {len(tier_a)} papers below should be included in your full-text screening.</strong></p>
"""

# Add TIER A papers
for idx, row in tier_a.iterrows():
    # Determine priority badge
    if row['ai_communication_relevance'] == 'highly_relevant':
        priority_badge = '<span class="badge badge-high">HIGH PRIORITY</span>'
    elif row['ai_communication_relevance'] == 'moderately_relevant':
        priority_badge = '<span class="badge badge-medium">MEDIUM PRIORITY</span>'
    else:
        priority_badge = '<span class="badge badge-low">LOW PRIORITY</span>'
    
    # Determine population badges
    pop_badges = []
    if row['dementia_relevance'] in ['highly_relevant', 'moderately_relevant']:
        pop_badges.append('<span class="badge badge-dementia">Dementia</span>')
    if row['aphasia_relevance'] in ['highly_relevant', 'moderately_relevant']:
        pop_badges.append('<span class="badge badge-aphasia">Aphasia</span>')
    
    html_content += f"""
    <div class="paper">
        <div class="paper-title">{idx + 1}. {row['title']}</div>
        <div class="paper-details">
            {priority_badge}
            {''.join(pop_badges)}
        </div>
        <div class="paper-details">
            <strong>Year:</strong> {row.get('year', 'N/A')} | 
            <strong>Journal:</strong> {row.get('journal', 'N/A')[:50]}
        </div>
        <div class="paper-details">
            <strong>AI Involvement:</strong> {row['ai_involvement']} | 
            <strong>Technology:</strong> {row.get('primary_technology', 'N/A')}
        </div>
        <div class="paper-details">
            <strong>Communication Domain:</strong> {row.get('communication_domain', 'N/A')}
        </div>
        <div class="paper-details">
            <strong>DOI:</strong> {row.get('doi', 'N/A')}
        </div>
    </div>
"""

# Add TIER B section
html_content += f"""
    <h2>⚠️ TIER B Papers (Unclear AI - Manual Review Needed)</h2>
    <p><strong>{len(tier_b)} papers need manual review to confirm AI involvement.</strong></p>
    <p>For each paper below, check if it uses AI/ML techniques (machine learning, NLP, speech recognition, computer vision, etc.)</p>
"""

for idx, row in tier_b.iterrows():
    html_content += f"""
    <div class="checklist-paper">
        <div class="paper-title">{idx + 1}. {row['title']}</div>
        <div class="paper-details">
            <strong>Year:</strong> {row.get('year', 'N/A')} | 
            <strong>Technology:</strong> {row.get('primary_technology', 'N/A')}
        </div>
        <div class="paper-details">
            <strong>Dementia:</strong> {row['dementia_relevance']} | 
            <strong>Aphasia:</strong> {row['aphasia_relevance']}
        </div>
        <div class="paper-details">
            <strong>Notes:</strong> {row.get('extraction_notes', 'N/A')[:200]}...
        </div>
        <div class="paper-details" style="margin-top: 10px;">
            <strong>✏️ Review Decision:</strong> 
            <input type="checkbox"> Include (has AI)  
            <input type="checkbox"> Exclude (no AI)  
            <input type="checkbox"> Unsure (check full text)
        </div>
    </div>
"""

html_content += """
    <div class="summary-box" style="margin-top: 40px;">
        <h2>📚 Files Created</h2>
        <ul>
            <li><strong>OPTION_A_TIER_A_confirmed_ai.csv</strong> - All TIER A papers (Excel/CSV format)</li>
            <li><strong>OPTION_A_TIER_B_CHECKLIST.csv</strong> - TIER B checklist for manual review</li>
            <li><strong>OPTION_A_RESULTS.html</strong> - This human-readable report</li>
            <li><strong>OPTION_A_RESULTS.txt</strong> - Simple text version</li>
        </ul>
    </div>
    
    <p style="text-align: center; color: #666; margin-top: 40px;">
        Generated on """ + datetime.now().strftime("%Y-%m-%d %H:%M") + """
    </p>
</body>
</html>
"""

with open('OPTION_A_RESULTS.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("  ✓ Created OPTION_A_RESULTS.html (open in any browser)")

# ----------------------------------------------------------------------------
# 2. SIMPLE TEXT REPORT
# ----------------------------------------------------------------------------

txt_content = f"""
================================================================================
OPTION A FILTERING RESULTS
================================================================================
Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}

SUMMARY
================================================================================
TIER A (Confirmed AI):           {len(tier_a)} papers
TIER B (Unclear AI - review):    {len(tier_b)} papers
Original approach:               {len(original)} papers
Papers added by Option A:        +{len(tier_a) - len(original)}

TIER A BREAKDOWN
================================================================================
By Priority:
  HIGH priority (highly relevant communication):    {len(priority_high)} papers
  MEDIUM priority (moderately relevant):            {len(priority_medium)} papers
  LOW priority (marginal communication):            {len(priority_low)} papers

By Population:
  Dementia-focused:                                 {len(dementia)} papers
  Aphasia-focused:                                  {len(aphasia)} papers
  Both populations:                                 {len(both)} papers

NEXT STEPS
================================================================================
1. Full-text screening: Include ALL {len(tier_a)} TIER A papers
2. Manual review: Check {len(tier_b)} TIER B papers for AI
   - Look for: machine learning, NLP, speech recognition, computer vision
   - Look for: adaptive systems, intelligent algorithms, neural networks
   - Expected outcome: ~{int(len(tier_b) * 0.3)} papers with confirmed AI
3. Total for screening: ~{len(tier_a) + int(len(tier_b) * 0.3)} papers
4. Expected final set: ~{int((len(tier_a) + int(len(tier_b) * 0.3)) * 0.7)} papers after full-text screening

================================================================================
TIER A PAPERS (CONFIRMED AI - INCLUDE ALL IN FULL-TEXT SCREENING)
================================================================================

"""

for idx, row in tier_a.iterrows():
    priority = "HIGH" if row['ai_communication_relevance'] == 'highly_relevant' else \
               "MEDIUM" if row['ai_communication_relevance'] == 'moderately_relevant' else "LOW"
    
    pop = []
    if row['dementia_relevance'] in ['highly_relevant', 'moderately_relevant']:
        pop.append('Dementia')
    if row['aphasia_relevance'] in ['highly_relevant', 'moderately_relevant']:
        pop.append('Aphasia')
    
    txt_content += f"""
{idx + 1}. {row['title']}

   Priority: {priority}
   Population: {', '.join(pop)}
   Year: {row.get('year', 'N/A')}
   AI Involvement: {row['ai_involvement']}
   Technology: {row.get('primary_technology', 'N/A')}
   Communication Domain: {row.get('communication_domain', 'N/A')}
   DOI: {row.get('doi', 'N/A')}
   
"""

txt_content += f"""
================================================================================
TIER B PAPERS (UNCLEAR AI - MANUAL REVIEW NEEDED)
================================================================================
Review each paper below and mark YES if it uses AI/ML techniques.
{len(tier_b)} papers total.

"""

for idx, row in tier_b.iterrows():
    txt_content += f"""
{idx + 1}. {row['title']}

   Year: {row.get('year', 'N/A')}
   Technology: {row.get('primary_technology', 'N/A')}
   Dementia relevance: {row['dementia_relevance']}
   Aphasia relevance: {row['aphasia_relevance']}
   Notes: {row.get('extraction_notes', 'N/A')[:150]}...
   
   [ ] Include (has AI)    [ ] Exclude (no AI)    [ ] Unsure
   
"""

txt_content += """
================================================================================
FILES CREATED
================================================================================
CSV Files (for Excel/data analysis):
  - OPTION_A_TIER_A_confirmed_ai.csv
  - OPTION_A_TIER_B_CHECKLIST.csv
  - OPTION_A_HIGH_PRIORITY.csv
  - OPTION_A_dementia_focused.csv
  - OPTION_A_aphasia_focused.csv

Human-Readable Files (no Excel needed):
  - OPTION_A_RESULTS.html (open in any web browser)
  - OPTION_A_RESULTS.txt (this file - open in any text editor)

================================================================================
"""

with open('OPTION_A_RESULTS.txt', 'w', encoding='utf-8') as f:
    f.write(txt_content)

print("  ✓ Created OPTION_A_RESULTS.txt (open in any text editor)")

# ----------------------------------------------------------------------------
# 3. MARKDOWN REPORT (for GitHub, documentation)
# ----------------------------------------------------------------------------

md_content = f"""# Option A Filtering Results

**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M")}

## Summary

| Metric | Count |
|--------|-------|
| TIER A (Confirmed AI) | {len(tier_a)} papers |
| TIER B (Unclear AI - review needed) | {len(tier_b)} papers |
| Original approach | {len(original)} papers |
| Papers added by Option A | +{len(tier_a) - len(original)} |

## TIER A Breakdown

### By Priority
- **HIGH priority** (highly relevant communication): {len(priority_high)} papers
- **MEDIUM priority** (moderately relevant): {len(priority_medium)} papers
- **LOW priority** (marginal communication): {len(priority_low)} papers

### By Population
- **Dementia-focused**: {len(dementia)} papers
- **Aphasia-focused**: {len(aphasia)} papers
- **Both populations**: {len(both)} papers

## Next Steps

1. **Full-text screening**: Include ALL {len(tier_a)} TIER A papers
2. **Manual review**: Check {len(tier_b)} TIER B papers for AI involvement
   - Look for: machine learning, NLP, speech recognition, computer vision
   - Expected outcome: ~{int(len(tier_b) * 0.3)} papers with confirmed AI
3. **Total for screening**: ~{len(tier_a) + int(len(tier_b) * 0.3)} papers

## TIER A Papers (Include in Full-Text Screening)

"""

for idx, row in tier_a.iterrows():
    priority = "🔴 HIGH" if row['ai_communication_relevance'] == 'highly_relevant' else \
               "🟡 MEDIUM" if row['ai_communication_relevance'] == 'moderately_relevant' else "⚪ LOW"
    
    pop = []
    if row['dementia_relevance'] in ['highly_relevant', 'moderately_relevant']:
        pop.append('Dementia')
    if row['aphasia_relevance'] in ['highly_relevant', 'moderately_relevant']:
        pop.append('Aphasia')
    
    md_content += f"""
### {idx + 1}. {row['title']}

- **Priority**: {priority}
- **Population**: {', '.join(pop)}
- **Year**: {row.get('year', 'N/A')}
- **AI Involvement**: {row['ai_involvement']}
- **Technology**: {row.get('primary_technology', 'N/A')}
- **DOI**: {row.get('doi', 'N/A')}

"""

md_content += f"""
## TIER B Papers (Manual Review Needed)

{len(tier_b)} papers with unclear AI involvement. Review each to confirm if it uses AI/ML techniques.

"""

with open('OPTION_A_RESULTS.md', 'w', encoding='utf-8') as f:
    f.write(md_content)

print("  ✓ Created OPTION_A_RESULTS.md (markdown format)")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n[9/9] Summary...")

print("\n" + "="*80)
print("✅ SUCCESS!")
print("="*80)

print(f"""
📊 RESULTS:
   • TIER A: {len(tier_a)} papers (confirmed AI)
   • TIER B: {len(tier_b)} papers (manual review needed)
   • Expected total: ~{len(tier_a) + int(len(tier_b) * 0.3)} papers for screening

📁 FILES CREATED:

CSV Files (Excel/spreadsheets):
   ✓ OPTION_A_TIER_A_confirmed_ai.csv
   ✓ OPTION_A_TIER_B_CHECKLIST.csv
   ✓ OPTION_A_HIGH_PRIORITY.csv
   ✓ OPTION_A_dementia_focused.csv
   ✓ OPTION_A_aphasia_focused.csv

Human-Readable Files (no special software needed):
   ✓ OPTION_A_RESULTS.html ← Open in any web browser
   ✓ OPTION_A_RESULTS.txt ← Open in Notepad/TextEdit
   ✓ OPTION_A_RESULTS.md ← For documentation/GitHub

🌐 TO VIEW RESULTS:
   1. Download OPTION_A_RESULTS.html
   2. Double-click to open in your web browser
   3. No Excel or coding needed!

🔄 NEXT STEPS:
   1. Download all files (click folder icon 📁)
   2. Open OPTION_A_RESULTS.html for easy reading
   3. Full-text screen all TIER A papers
   4. Manual review TIER B checklist (3-4 hours)
""")

print("="*80)
print("🎉 FILTERING COMPLETE!")
print("="*80)

In [ ]:
"""
CSV TO HUMAN-READABLE CONVERTER
================================
Run this AFTER option_a_filtering to convert CSV files to HTML/TXT
Works with existing OPTION_A_*.csv files
"""

import pandas as pd
import os
from datetime import datetime

print("="*80)
print("CSV TO HUMAN-READABLE CONVERTER")
print("="*80)

# ============================================================================
# CHECK FOR REQUIRED FILES
# ============================================================================

print("\n[1/3] Checking for CSV files...")

required_files = {
    'tier_a': 'OPTION_A_TIER_A_confirmed_ai.csv',
    'tier_b': 'OPTION_A_TIER_B_CHECKLIST.csv'
}

files_found = {}
missing = []

for key, filename in required_files.items():
    if os.path.exists(filename):
        print(f"  ✓ Found: {filename}")
        files_found[key] = filename
    else:
        print(f"  ✗ Missing: {filename}")
        missing.append(filename)

if missing:
    print(f"\n  ❌ ERROR: Required files not found: {missing}")
    print(f"\n  Please run the Option A filtering script first!")
    import sys
    sys.exit(1)

print(f"  ✓ All required files found")

# ============================================================================
# LOAD DATA
# ============================================================================

print("\n[2/3] Loading data...")

tier_a = pd.read_csv(files_found['tier_a'])
tier_b = pd.read_csv(files_found['tier_b'])

print(f"  ✓ TIER A: {len(tier_a)} papers")
print(f"  ✓ TIER B: {len(tier_b)} papers")

# Calculate stats
priority_high = tier_a[tier_a['ai_communication_relevance'] == 'highly_relevant'] if 'ai_communication_relevance' in tier_a.columns else pd.DataFrame()
priority_medium = tier_a[tier_a['ai_communication_relevance'] == 'moderately_relevant'] if 'ai_communication_relevance' in tier_a.columns else pd.DataFrame()
priority_low = tier_a[tier_a['ai_communication_relevance'].isin(['marginally_relevant', 'not_relevant'])] if 'ai_communication_relevance' in tier_a.columns else pd.DataFrame()

# ============================================================================
# CREATE HTML REPORT
# ============================================================================

print("\n[3/3] Creating human-readable outputs...")

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Option A Results - Easy to Read</title>
    <style>
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            max-width: 1200px;
            margin: 20px auto;
            padding: 20px;
            background-color: #f8f9fa;
            line-height: 1.6;
        }}
        h1 {{
            color: #2c3e50;
            border-bottom: 3px solid #3498db;
            padding-bottom: 10px;
        }}
        h2 {{
            color: #34495e;
            margin-top: 30px;
            border-left: 4px solid #3498db;
            padding-left: 10px;
        }}
        .summary-box {{
            background-color: white;
            border: 1px solid #dee2e6;
            border-radius: 8px;
            padding: 25px;
            margin: 20px 0;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }}
        .stat {{
            display: inline-block;
            margin: 10px 15px 10px 0;
            padding: 12px 20px;
            background-color: #3498db;
            color: white;
            border-radius: 6px;
            font-weight: bold;
            font-size: 16px;
        }}
        .paper {{
            background-color: white;
            border-left: 4px solid #3498db;
            padding: 20px;
            margin: 20px 0;
            border-radius: 6px;
            box-shadow: 0 2px 5px rgba(0,0,0,0.08);
            transition: box-shadow 0.3s;
        }}
        .paper:hover {{
            box-shadow: 0 4px 12px rgba(0,0,0,0.15);
        }}
        .paper-title {{
            font-size: 18px;
            font-weight: bold;
            color: #2c3e50;
            margin-bottom: 12px;
            line-height: 1.4;
        }}
        .paper-details {{
            font-size: 14px;
            color: #6c757d;
            margin: 8px 0;
        }}
        .badge {{
            display: inline-block;
            padding: 4px 10px;
            border-radius: 4px;
            font-size: 12px;
            font-weight: bold;
            margin-right: 6px;
            margin-bottom: 4px;
        }}
        .badge-high {{ background-color: #dc3545; color: white; }}
        .badge-medium {{ background-color: #fd7e14; color: white; }}
        .badge-low {{ background-color: #6c757d; color: white; }}
        .badge-dementia {{ background-color: #6f42c1; color: white; }}
        .badge-aphasia {{ background-color: #20c997; color: white; }}
        .next-steps {{
            background-color: #fff3cd;
            border: 2px solid #ffc107;
            border-radius: 8px;
            padding: 25px;
            margin: 20px 0;
        }}
        .next-steps ol {{
            margin: 15px 0;
            padding-left: 25px;
        }}
        .next-steps li {{
            margin: 10px 0;
            font-size: 16px;
        }}
        .checklist-paper {{
            background-color: #fffbeb;
            border-left: 4px solid #fbbf24;
            padding: 20px;
            margin: 20px 0;
            border-radius: 6px;
        }}
        .review-checkboxes {{
            margin-top: 15px;
            padding: 12px;
            background-color: white;
            border-radius: 4px;
            border: 1px solid #e5e7eb;
        }}
        .review-checkboxes label {{
            display: inline-block;
            margin-right: 20px;
            font-size: 14px;
            cursor: pointer;
        }}
        @media print {{
            .paper {{
                page-break-inside: avoid;
            }}
        }}
    </style>
</head>
<body>
    <h1>📚 Systematic Review Filtering Results</h1>
    <p style="color: #6c757d; font-size: 14px;">
        Option A Approach: AI-focused filtering without hard communication filter<br>
        Generated: {datetime.now().strftime("%B %d, %Y at %I:%M %p")}
    </p>
    
    <div class="summary-box">
        <h2>📊 Summary</h2>
        <div class="stat">TIER A: {len(tier_a)} papers</div>
        <div class="stat" style="background-color: #fbbf24; color: #000;">TIER B: {len(tier_b)} papers</div>
        <div class="stat" style="background-color: #10b981;">Expected: ~{len(tier_a) + int(len(tier_b) * 0.3)} total</div>
        
        <h3 style="margin-top: 25px;">TIER A Priorities</h3>
        <div class="stat" style="background-color: #dc3545;">HIGH: {len(priority_high)}</div>
        <div class="stat" style="background-color: #fd7e14;">MEDIUM: {len(priority_medium)}</div>
        <div class="stat" style="background-color: #6c757d;">LOW: {len(priority_low)}</div>
    </div>
    
    <div class="next-steps">
        <h2>🎯 What to Do Next</h2>
        <ol>
            <li><strong>Full-Text Screening:</strong> Include ALL {len(tier_a)} papers from TIER A below</li>
            <li><strong>Manual Review:</strong> Review the {len(tier_b)} TIER B papers (scroll down) to check for AI
                <ul>
                    <li>Look for keywords: machine learning, NLP, speech recognition, computer vision</li>
                    <li>Look for: adaptive systems, intelligent algorithms, neural networks</li>
                    <li>Budget: 3-4 hours (about 5 minutes per paper)</li>
                </ul>
            </li>
            <li><strong>Expected Outcome:</strong> About {int(len(tier_b) * 0.3)} more papers from TIER B will have confirmed AI</li>
            <li><strong>Total for Screening:</strong> Approximately {len(tier_a) + int(len(tier_b) * 0.3)} papers</li>
        </ol>
    </div>
    
    <h2>✅ TIER A: Confirmed AI Papers ({len(tier_a)} papers)</h2>
    <p><strong>Include ALL of these papers in your full-text screening.</strong></p>
    <p style="color: #6c757d; margin-bottom: 30px;">Papers are listed with priority levels to help you plan your review order.</p>
"""

# Add TIER A papers
for idx, row in tier_a.iterrows():
    # Priority badge
    if 'ai_communication_relevance' in row and row['ai_communication_relevance'] == 'highly_relevant':
        priority_badge = '<span class="badge badge-high">🔴 HIGH PRIORITY</span>'
    elif 'ai_communication_relevance' in row and row['ai_communication_relevance'] == 'moderately_relevant':
        priority_badge = '<span class="badge badge-medium">🟡 MEDIUM PRIORITY</span>'
    else:
        priority_badge = '<span class="badge badge-low">⚪ LOW PRIORITY</span>'
    
    # Population badges
    pop_badges = []
    if 'dementia_relevance' in row and row['dementia_relevance'] in ['highly_relevant', 'moderately_relevant']:
        pop_badges.append('<span class="badge badge-dementia">Dementia</span>')
    if 'aphasia_relevance' in row and row['aphasia_relevance'] in ['highly_relevant', 'moderately_relevant']:
        pop_badges.append('<span class="badge badge-aphasia">Aphasia</span>')
    
    html_content += f"""
    <div class="paper">
        <div class="paper-title">{idx + 1}. {row.get('title', 'No title')}</div>
        <div class="paper-details">
            {priority_badge}
            {''.join(pop_badges)}
        </div>
        <div class="paper-details">
            <strong>Year:</strong> {row.get('year', 'N/A')} &nbsp;|&nbsp; 
            <strong>Journal:</strong> {str(row.get('journal', 'N/A'))[:60]}
        </div>
        <div class="paper-details">
            <strong>AI Type:</strong> {row.get('ai_involvement', 'N/A')} &nbsp;|&nbsp; 
            <strong>Technology:</strong> {row.get('primary_technology', 'N/A')}
        </div>
        <div class="paper-details">
            <strong>Communication Focus:</strong> {row.get('communication_domain', 'N/A')}
        </div>
        {f'<div class="paper-details"><strong>DOI:</strong> <a href="https://doi.org/{row["doi"]}" target="_blank">{row["doi"]}</a></div>' if pd.notna(row.get('doi')) and row.get('doi') != 'N/A' else ''}
    </div>
"""

# Add TIER B section
html_content += f"""
    <h2 style="margin-top: 50px;">⚠️ TIER B: Papers Needing Manual Review ({len(tier_b)} papers)</h2>
    <p><strong>These papers have unclear AI involvement.</strong> Please review each one to determine if it uses AI/ML techniques.</p>
    <p style="background-color: #fff3cd; padding: 15px; border-radius: 4px; border-left: 4px solid #ffc107;">
        <strong>How to review:</strong> For each paper, read the title and notes. Look for mentions of:
        machine learning, NLP, speech recognition, computer vision, adaptive systems, intelligent algorithms, 
        neural networks, deep learning, or any computational decision-making. Mark your decision using the checkboxes.
    </p>
"""

for idx, row in tier_b.iterrows():
    html_content += f"""
    <div class="checklist-paper">
        <div class="paper-title">{idx + 1}. {row.get('title', 'No title')}</div>
        <div class="paper-details">
            <strong>Year:</strong> {row.get('year', 'N/A')} &nbsp;|&nbsp; 
            <strong>Technology Listed:</strong> {row.get('primary_technology', 'N/A')}
        </div>
        <div class="paper-details">
            <strong>Population:</strong> 
            Dementia: {row.get('dementia_relevance', 'N/A')} &nbsp;|&nbsp; 
            Aphasia: {row.get('aphasia_relevance', 'N/A')}
        </div>
        <div class="paper-details">
            <strong>Notes:</strong> {str(row.get('extraction_notes', 'No notes available'))[:250]}...
        </div>
        <div class="review-checkboxes">
            <strong>✏️ Your Review Decision:</strong><br>
            <label><input type="checkbox"> ✅ INCLUDE (has AI/ML)</label>
            <label><input type="checkbox"> ❌ EXCLUDE (no AI)</label>
            <label><input type="checkbox"> ❓ UNSURE (check full text)</label>
        </div>
    </div>
"""

html_content += f"""
    <div class="summary-box" style="margin-top: 50px;">
        <h2>📁 Files Available</h2>
        <p>You have both machine-readable and human-readable versions:</p>
        <ul style="font-size: 15px; line-height: 2;">
            <li><strong>OPTION_A_TIER_A_confirmed_ai.csv</strong> - For Excel/data analysis</li>
            <li><strong>OPTION_A_TIER_B_CHECKLIST.csv</strong> - For Excel/data analysis</li>
            <li><strong>OPTION_A_RESULTS.html</strong> - This file (open in web browser)</li>
            <li><strong>OPTION_A_RESULTS.txt</strong> - Simple text version (open in Notepad)</li>
        </ul>
        <p style="margin-top: 20px; color: #6c757d;">
            💡 <strong>Tip:</strong> You can print this page (Ctrl+P / Cmd+P) or save as PDF for offline review.
        </p>
    </div>
    
    <div style="text-align: center; color: #adb5bd; margin-top: 50px; padding: 20px; border-top: 1px solid #dee2e6;">
        <p>Generated with Option A filtering approach</p>
        <p>No hard filter on ai_communication_relevance</p>
        <p style="margin-top: 10px;">{datetime.now().strftime("%Y-%m-%d %H:%M")}</p>
    </div>
</body>
</html>
"""

with open('OPTION_A_RESULTS.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("  ✓ Created: OPTION_A_RESULTS.html")

# ============================================================================
# CREATE TEXT VERSION
# ============================================================================

txt_content = f"""
{'='*80}
OPTION A FILTERING RESULTS - EASY TO READ VERSION
{'='*80}
Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}

SUMMARY
{'='*80}
TIER A (Confirmed AI - include in screening):  {len(tier_a)} papers
TIER B (Unclear AI - needs manual review):     {len(tier_b)} papers
Expected additional papers from TIER B:        ~{int(len(tier_b) * 0.3)} papers
TOTAL expected for full-text screening:        ~{len(tier_a) + int(len(tier_b) * 0.3)} papers

TIER A PRIORITIES
{'='*80}
HIGH priority (direct communication focus):    {len(priority_high)} papers
MEDIUM priority (indirect communication):      {len(priority_medium)} papers
LOW priority (minimal communication focus):    {len(priority_low)} papers

NEXT STEPS
{'='*80}
1. Full-text screening: Include ALL {len(tier_a)} TIER A papers listed below
2. Manual review: Check {len(tier_b)} TIER B papers for AI (3-4 hours)
   - Look for: machine learning, NLP, speech recognition, computer vision
   - Look for: adaptive systems, intelligent algorithms, neural networks
3. Expected outcome: ~{int(len(tier_b) * 0.3)} additional papers from TIER B
4. Total for screening: ~{len(tier_a) + int(len(tier_b) * 0.3)} papers

{'='*80}
TIER A PAPERS (CONFIRMED AI - INCLUDE ALL IN FULL-TEXT SCREENING)
{'='*80}

"""

for idx, row in tier_a.iterrows():
    priority = "HIGH" if 'ai_communication_relevance' in row and row['ai_communication_relevance'] == 'highly_relevant' else \
               "MEDIUM" if 'ai_communication_relevance' in row and row['ai_communication_relevance'] == 'moderately_relevant' else "LOW"
    
    pop = []
    if 'dementia_relevance' in row and row['dementia_relevance'] in ['highly_relevant', 'moderately_relevant']:
        pop.append('Dementia')
    if 'aphasia_relevance' in row and row['aphasia_relevance'] in ['highly_relevant', 'moderately_relevant']:
        pop.append('Aphasia')
    
    txt_content += f"""
{'─'*80}
{idx + 1}. {row.get('title', 'No title')}

Priority: {priority}
Population: {', '.join(pop) if pop else 'N/A'}
Year: {row.get('year', 'N/A')}
Journal: {row.get('journal', 'N/A')}
AI Involvement: {row.get('ai_involvement', 'N/A')}
Technology: {row.get('primary_technology', 'N/A')}
Communication Domain: {row.get('communication_domain', 'N/A')}
DOI: {row.get('doi', 'N/A')}

"""

txt_content += f"""
{'='*80}
TIER B PAPERS (UNCLEAR AI - MANUAL REVIEW NEEDED)
{'='*80}
Review each paper below. Mark [ ] with X if it has AI/ML techniques.
{len(tier_b)} papers total - budget 3-4 hours.

"""

for idx, row in tier_b.iterrows():
    txt_content += f"""
{'─'*80}
{idx + 1}. {row.get('title', 'No title')}

Year: {row.get('year', 'N/A')}
Technology: {row.get('primary_technology', 'N/A')}
Dementia: {row.get('dementia_relevance', 'N/A')}
Aphasia: {row.get('aphasia_relevance', 'N/A')}
Notes: {str(row.get('extraction_notes', 'No notes'))[:200]}...

Review Decision:
[ ] INCLUDE (has AI/ML techniques)
[ ] EXCLUDE (no AI)
[ ] UNSURE (need to check full text)

"""

txt_content += f"""
{'='*80}
FILES CREATED
{'='*80}
Machine-readable (for Excel/analysis):
  - OPTION_A_TIER_A_confirmed_ai.csv
  - OPTION_A_TIER_B_CHECKLIST.csv

Human-readable (no special software):
  - OPTION_A_RESULTS.html (open in web browser - RECOMMENDED)
  - OPTION_A_RESULTS.txt (this file - open in any text editor)

TIP: Open the HTML file for an easier reading experience with formatting!

{'='*80}
END OF REPORT
{'='*80}
"""

with open('OPTION_A_RESULTS.txt', 'w', encoding='utf-8') as f:
    f.write(txt_content)

print("  ✓ Created: OPTION_A_RESULTS.txt")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*80)
print("✅ CONVERSION COMPLETE!")
print("="*80)

print(f"""
📁 Human-Readable Files Created:

1. OPTION_A_RESULTS.html ({os.path.getsize('OPTION_A_RESULTS.html') / 1024:.0f} KB)
   → Open in ANY web browser (Chrome, Firefox, Safari, Edge)
   → Formatted with colors and easy navigation
   → Can be printed or saved as PDF
   
2. OPTION_A_RESULTS.txt ({os.path.getsize('OPTION_A_RESULTS.txt') / 1024:.0f} KB)
   → Open in Notepad, TextEdit, or any text editor
   → Simple plain text format
   → Easy to read and share

🌐 TO VIEW:
   • Double-click OPTION_A_RESULTS.html
   • Opens in your default web browser
   • No Excel or coding knowledge needed!

📄 YOUR CSV FILES STILL AVAILABLE:
   • OPTION_A_TIER_A_confirmed_ai.csv (for data analysis)
   • OPTION_A_TIER_B_CHECKLIST.csv (for data analysis)

✨ NOW YOU HAVE BOTH:
   • CSV files for statistical analysis
   • HTML/TXT files for easy reading
""")

print("="*80)